# 4.9 Training clinic: one classifier, six decisions

Notebook 4.3 trained a convolutional detector on synthetic seismograms. Today the records are real: **miniPNW**, a labeled subset of the Pacific Northwest AI-ready benchmark ([Ni et al., 2023](https://doi.org/10.26443/seismica.v2i1.368)) that you met in [2.11](../Chapter2-DataManipulation/2.11_feature_engineering.ipynb). Each record is three components of ground velocity (north, east, vertical) at one station, labeled by the Pacific Northwest Seismic Network analysts as an **earthquake**, an **explosion** (mostly quarry blasts), a **surface event** (rockfall, avalanche, debris flow), a **sonic boom** or **thunder**.

The task: from 30 s of three-component ground motion, name the source.

The network is fixed and small. What changes from station to station is one decision a modeler makes while training:

| Station | Decision | Question you answer with a number |
|---|---|---|
| 1 | Train, validation and test sets; leakage | How much does the test score change with the way you split? |
| 2 | Starting weights | What happens if every weight starts at zero? |
| 3 | Learning rate and batch size | Which pairs train, and which do not? |
| 4 | Model size and amount of data | Would more data or a bigger network help more? |
| 5 | Data augmentation | Does the network still work when the arrival is not where it was in training? |
| 6 | The loss | Which loss finds the rare classes (sonic booms, thunder)? |

**How to work.** Each team takes two stations (the instructor assigns them). Run sections 0 to 2 first: they load the data and define the training code that every station uses. In each station:

1. **Predict** the result before you run anything, and write the prediction in the cell provided.
2. **Write** the short cell marked *Your turn*. If you leave it empty or it fails, a hidden cell uses a reference answer so the rest of the station still runs.
3. **Run** and compare with your prediction.
4. **Write one rule** for training a network, backed by the number you measured, in the format of section 9. These rules become the class rubric at the end of the session.

Every station runs in one to three minutes on a laptop CPU.

## 0. Setup

In [ ]:
import os
import sys
import time

try:
    import h5py
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from scipy.signal import decimate
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import balanced_accuracy_score, confusion_matrix, recall_score
    from sklearn.model_selection import GroupShuffleSplit
except ModuleNotFoundError as err:
    raise ModuleNotFoundError(
        f"'{err.name}' is not installed in this kernel ({sys.executable}).\n"
        "Use the course environment. In VS Code: 'Select Kernel' (top right), then 'Python Environments', "
        "then the one in .pixi/envs/default, or 'Python (mlgeo-book pixi)'. "
        "In a terminal, from the mlgeo-book folder: pixi install, then pixi run jupyter lab.") from None

device = torch.device("cpu")      # every station is sized for a laptop CPU
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
SEED = 469                        # the seed of every random choice in this notebook
rng = np.random.default_rng(SEED)
plt.rcParams.update({"font.size": 12})
print("PyTorch", torch.__version__, "| threads:", torch.get_num_threads())

In [ ]:
# Helpers for the "Your turn" cells.
# check(): compare a number with a reference; use_or_reference(): test a function you wrote.
# If your answer is missing or does not pass, the reference is used so the cells below still run.
def check(name, ns, reference, hint, tol=1e-5):
    yours = ns.get(name)
    if yours is None:
        print(f"{name}: not answered yet. Hint: {hint}\n  Using the reference result below.")
    else:
        try:
            ok = np.allclose(np.asarray(yours, dtype=float), np.asarray(reference, dtype=float), atol=tol)
        except Exception:
            ok = False
        if ok:
            print(f"{name}: correct")
            return
        print(f"{name}: not the expected result. Hint: {hint}\n  Your answer is kept as {name}_yours; "
              "the reference is used below.")
        ns[f"{name}_yours"] = yours
    ns[name] = reference


def use_or_reference(name, ns, reference_fn, test, hint):
    yours = ns.get(name)
    if yours is None:
        print(f"{name}: not written yet. Hint: {hint}\n  Using the reference function below.")
        ns[name] = reference_fn
        return
    try:
        ok = bool(test(yours))
    except Exception as err:
        ok = False
        print(f"{name}: raised {type(err).__name__}: {err}")
    if ok:
        print(f"{name}: passes the test, yours is used")
    else:
        print(f"{name}: does not pass the test. Hint: {hint}\n  Yours is kept as {name}_yours; "
              "the reference is used below.")
        ns[f"{name}_yours"] = yours
        ns[name] = reference_fn

## 1. The data: 1,720 records from the Pacific Northwest

The two miniPNW files are the ones notebooks 2.11 and 4.3 use, in the same folder. If they are not on your disk, download them **at home** (671 MB; thirty laptops downloading at once in class stall the network). From the top of your `mlgeo-book` folder:

```bash
mkdir -p book/Chapter2-DataManipulation/data
cd book/Chapter2-DataManipulation/data
BASE=https://dasway.ess.washington.edu/shared/niyiyu/PNW-ML
curl -fL -C - -O $BASE/miniPNW_metadata.csv
curl -fL -C - -O $BASE/miniPNW_waveforms.hdf5
```

The cell below reads one 30 s window per record, from 5 s before the analyst's P pick to 25 s after, and a second 30 s window that ends before the first one starts (background noise, used in station 5). It keeps 50 samples per second (low-pass filtered, then one sample in two), and saves the windows in a 60 MB file so the next run takes a second.

In [ ]:
DATA = os.path.join("..", "Chapter2-DataManipulation", "data")
META = os.path.join(DATA, "miniPNW_metadata.csv")
WAVES = os.path.join(DATA, "miniPNW_waveforms.hdf5")
CACHE = os.path.join(DATA, "miniPNW_clinic_windows.npz")
FS = 50.0                        # samples per second after decimation
PRE, WIN = 500, 3000             # at 100 Hz: 5 s before the P pick, 30 s window

if not os.path.exists(META) or not os.path.exists(WAVES):
    raise FileNotFoundError("miniPNW files not found in " + os.path.abspath(DATA) +
                            ". Download them with the commands in the cell above, then rerun.")

meta = pd.read_csv(META)
meta = meta[meta["trace_P_arrival_sample"].notna()].reset_index(drop=True)
print(len(meta), "records with a P pick")

if os.path.exists(CACHE):
    z = np.load(CACHE)
    X_all, N_all = z["X"], z["N"]
else:
    t0 = time.time()
    X_all = np.zeros((len(meta), 3, WIN // 2), dtype=np.float32)
    N_all = np.zeros_like(X_all)
    with h5py.File(WAVES, "r") as f:
        for i, name in enumerate(meta["trace_name"]):
            bucket, arr = name.split("$")
            row = int(arr.split(",")[0])
            p = int(meta.loc[i, "trace_P_arrival_sample"])
            rec = f["/data/" + bucket][row].astype(np.float64)          # (3, 15001): N, E, Z at 100 Hz
            sig = rec[:, p - PRE: p - PRE + WIN]
            noi = rec[:, p - PRE - WIN: p - PRE]
            X_all[i] = decimate(sig, 2, axis=1, zero_phase=True)
            N_all[i] = decimate(noi, 2, axis=1, zero_phase=True)
    np.savez_compressed(CACHE, X=X_all, N=N_all)
    print(f"read and cached in {time.time() - t0:.0f} s")

CLASSES = ["earthquake", "explosion", "surface_event", "sonic_boom", "thunder"]
y_all = meta["source_type"].map({c: i for i, c in enumerate(CLASSES)}).to_numpy()
print("windows:", X_all.shape, "(records, components N E Z, samples at 50 Hz)")
print(meta["source_type"].value_counts().to_string())

Look before you train: where are the sources and the stations, and what does each class look like?

In [ ]:
fig = plt.figure(figsize=(14, 6))
ax = fig.add_subplot(1, 2, 1)
colors = dict(zip(CLASSES, ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00"]))
for c in CLASSES:
    m = meta[meta.source_type == c]
    ax.scatter(m.source_longitude_deg, m.source_latitude_deg, s=10, color=colors[c], label=f"{c} ({len(m)})")
st = meta.drop_duplicates("station_code")
ax.scatter(st.station_longitude_deg, st.station_latitude_deg, marker="^", s=18, color="k", label=f"stations ({len(st)})")
ax.set_xlabel("longitude (°)"); ax.set_ylabel("latitude (°)"); ax.set_aspect(1 / np.cos(np.radians(46)))
ax.set_title("Sources and stations, Pacific Northwest"); ax.legend(fontsize=9, loc="lower left")

t = np.arange(X_all.shape[2]) / FS - 5
for k, c in enumerate(CLASSES):
    a = fig.add_subplot(5, 2, 2 * k + 2)
    i = np.where(y_all == k)[0][0]
    a.plot(t, X_all[i, 2] / np.abs(X_all[i, 2]).max(), color=colors[c], lw=0.6)
    a.axvline(0, color="k", ls=":", lw=1)
    a.set_yticks([]); a.set_ylabel(c.replace("_", " "), rotation=0, ha="right", fontsize=10)
    if k < 4:
        a.set_xticks([])
a.set_xlabel("time after the P pick (s), vertical component")
fig.tight_layout()

**Scaling.** Each record is divided by the standard deviation of its own 30 s window (one number for the three components, so their relative size is kept). This uses no other record, so it cannot leak information between the training and test sets, and the network cannot use the absolute amplitude, which depends mostly on distance and magnitude.

In [ ]:
def scale_each(X):
    return (X / (X.std(axis=(1, 2), keepdims=True) + 1e-12)).astype(np.float32)

Xs = scale_each(X_all)
Ns = scale_each(N_all)
groups_event = meta["event_id"].to_numpy()
groups_station = meta["station_code"].to_numpy()
print("events:", len(set(groups_event)), "| stations:", len(set(groups_station)),
      "| events recorded at more than one station:", int((meta.event_id.value_counts() > 1).sum()))

## 2. The training code every station uses

Read this section; you do not need to change it. Two networks are available:

- `make_cnn`: three blocks of convolution and pooling over the 30 s, then an average over time and one score per class (notebook 4.3).
- `make_fcn`: a fully connected network on a table of 90 numbers per record: the logarithm of the root-mean-square amplitude of each component in each second (notebook 4.1).

`train` uses the training loop of 4.1: random starting weights drawn with a seed, Adam, cross-entropy, the mean loss over each batch, and the training and validation loss at every epoch. It keeps the weights of the epoch with the lowest validation loss.

The score is the **balanced accuracy**: the mean of the recall of each class. With 500 earthquakes and 94 thunder records, plain accuracy rewards a network that never says "thunder".

In [ ]:
def make_cnn(width=16, kernel=7, n_classes=5):
    w = width
    return nn.Sequential(
        nn.Conv1d(3, w, kernel, padding=kernel // 2), nn.ReLU(), nn.MaxPool1d(4),
        nn.Conv1d(w, 2 * w, kernel, padding=kernel // 2), nn.ReLU(), nn.MaxPool1d(4),
        nn.Conv1d(2 * w, 2 * w, kernel, padding=kernel // 2), nn.ReLU(),
        nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Linear(2 * w, n_classes))


class RMSTable(nn.Module):
    # 30 s x 3 components -> log RMS amplitude in each second: a table of 90 features
    def forward(self, x):
        b = x.shape[0]
        return torch.log(x.reshape(b, 3, 30, -1).pow(2).mean(-1).sqrt() + 1e-3).reshape(b, 90)


def make_fcn(width=64, n_classes=5):
    return nn.Sequential(RMSTable(), nn.Linear(90, width), nn.ReLU(), nn.Linear(width, n_classes))


def build(arch="cnn", seed=0, **kw):
    torch.manual_seed(seed)          # the starting weights are random numbers drawn with this seed
    return (make_cnn if arch == "cnn" else make_fcn)(**kw)


def n_weights(model):
    return sum(p.numel() for p in model.parameters())


def predict(model, X, batch=256):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(torch.from_numpy(X[i:i + batch])) for i in range(0, len(X), batch)]).argmax(1).numpy()


def score(model, X, y):
    return balanced_accuracy_score(y, predict(model, X))


def train(model, idx_tr, idx_va, X=None, y=None, lr=1e-3, batch=64, epochs=12, seed=0,
          loss_fn=None, augment=None, max_updates=None, verbose=False):
    X = Xs if X is None else X
    y = y_all if y is None else y
    loss_fn = loss_fn or nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    g = torch.Generator().manual_seed(seed)
    Xtr, ytr = torch.from_numpy(X[idx_tr]), torch.from_numpy(y[idx_tr])
    Xva, yva = torch.from_numpy(X[idx_va]), torch.from_numpy(y[idx_va])
    hist = {"train_loss": [], "val_loss": [], "val_score": [], "batch_loss": []}
    best, best_state, updates = np.inf, None, 0
    for ep in range(epochs):
        model.train()
        order = torch.randperm(len(Xtr), generator=g)
        tot = 0.0
        for i in range(0, len(order), batch):
            j = order[i:i + batch]
            xb = Xtr[j] if augment is None else augment(Xtr[j].clone())
            loss = loss_fn(model(xb), ytr[j])
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(j); updates += 1
            hist["batch_loss"].append(loss.item())
            if max_updates and updates >= max_updates:
                break
        model.eval()
        with torch.no_grad():
            va = nn.CrossEntropyLoss()(model(Xva), yva).item()
        hist["train_loss"].append(tot / min(len(order), (i + batch)))
        hist["val_loss"].append(va)
        hist["val_score"].append(score(model, X[idx_va], y[idx_va]))
        if va < best:
            best, best_state = va, {k: v.clone() for k, v in model.state_dict().items()}
        if verbose:
            print(f"epoch {ep + 1:2d}  train loss {hist['train_loss'][-1]:.3f}  "
                  f"val loss {va:.3f}  val balanced acc {hist['val_score'][-1]:.3f}")
        if max_updates and updates >= max_updates:
            break
    if best_state is not None:        # None if the loss was never finite
        model.load_state_dict(best_state)
    hist["updates"] = updates
    return hist


def plot_curves(histories, title="", ax=None):
    ax = ax or plt.subplots(figsize=(7, 4))[1]
    for (label, h), c in zip(histories.items(), plt.cm.tab10.colors):
        ep = np.arange(1, len(h["train_loss"]) + 1)
        ax.plot(ep, h["train_loss"], color=c, lw=2, label=f"{label} train")
        ax.plot(ep, h["val_loss"], color=c, lw=2, ls="--", label=f"{label} val")
    ax.set_xlabel("epoch"); ax.set_ylabel("cross-entropy loss"); ax.set_title(title, loc="left")
    ax.legend(fontsize=8, ncol=2)
    return ax


def split_by_group(groups, seed=SEED, val=0.15, test=0.15):
    # train / validation / test such that no group appears in two of them
    idx = np.arange(len(groups))
    a, te = next(GroupShuffleSplit(1, test_size=test, random_state=seed).split(idx, groups=groups))
    b, va = next(GroupShuffleSplit(1, test_size=val / (1 - test), random_state=seed).split(a, groups=groups[a]))
    return a[b], a[va], te


# the split every station uses unless it says otherwise: no event in two partitions
idx_tr, idx_va, idx_te = split_by_group(groups_event)
print("train / val / test records:", len(idx_tr), len(idx_va), len(idx_te))

t0 = time.time()
cnn = build("cnn", seed=0)
h0 = train(cnn, idx_tr, idx_va, verbose=True)
print(f"\nCNN, {n_weights(cnn):,} weights, trained in {time.time() - t0:.0f} s; "
      f"test balanced accuracy {score(cnn, Xs[idx_te], y_all[idx_te]):.3f}")
plot_curves({"CNN": h0}, "Reference run: the CNN, default settings")

In [ ]:
# The baseline without learning, and a classical model on the same split
majority = balanced_accuracy_score(y_all[idx_te], np.zeros(len(idx_te), dtype=int))
rms = RMSTable()(torch.from_numpy(Xs)).numpy()
logit = LogisticRegression(max_iter=2000).fit(rms[idx_tr], y_all[idx_tr])
print(f"always 'earthquake' (no learning):        {majority:.3f}")
print(f"logistic regression on the 90 RMS values: {balanced_accuracy_score(y_all[idx_te], logit.predict(rms[idx_te])):.3f}")
print(f"CNN reference run:                        {score(cnn, Xs[idx_te], y_all[idx_te]):.3f}")

## 3. Station 1: train, validation and test sets, and leakage

Three ways information about the test set can reach the model, each measured.

**(a) The split.** The network is meant to classify sources it has never seen, at stations that may be new. Records of one event at two stations look alike; records of one quarry at one station look alike over years. Compare three splits: records drawn at random, records grouped by event, records grouped by station.

*Prediction (write it before running):* which split gives the highest test score, and by how much?

In [ ]:
my_prediction_1a = ""   # e.g. "random split highest, about 0.05 above grouped by station"

### Your turn (1): a split grouped by station

Write `idx_tr_st, idx_va_st, idx_te_st` so that no station appears in two partitions. One line: the function `split_by_group` of section 2 takes the group label of each record.

In [ ]:
idx_tr_st, idx_va_st, idx_te_st = None, None, None   # your line here

In [ ]:
ref = split_by_group(groups_station)
ok = idx_tr_st is not None and set(groups_station[idx_tr_st]).isdisjoint(groups_station[idx_te_st]) \
     and set(groups_station[idx_tr_st]).isdisjoint(groups_station[idx_va_st])
if not ok:
    print("idx_*_st: missing, or a station appears in two partitions. Hint: split_by_group(groups_station)."
          "\n  Using the reference split below.")
    idx_tr_st, idx_va_st, idx_te_st = ref
else:
    print("station split: no station in two partitions")

perm = rng.permutation(len(y_all))
n_te, n_va = int(0.15 * len(perm)), int(0.15 * len(perm))
random_split = perm[n_te + n_va:], perm[n_te:n_te + n_va], perm[:n_te]

splits = {"random records": random_split, "grouped by event": (idx_tr, idx_va, idx_te),
          "grouped by station": (idx_tr_st, idx_va_st, idx_te_st)}
res1 = {}
for name, (a, b, c) in splits.items():
    s = []
    for seed in range(3):
        m = build("cnn", seed=seed)
        train(m, a, b, seed=seed)
        s.append(score(m, Xs[c], y_all[c]))
    res1[name] = s
    print(f"{name:20s} test balanced accuracy {np.mean(s):.3f} ± {np.std(s):.3f}  (3 seeds)")

**(b) A label hidden in the bookkeeping.** The analyst's P pick gives the position of the window in the 150 s record. Train a logistic regression on that one number, the sample index of the P pick, and nothing else.

In [ ]:
pick = meta[["trace_P_arrival_sample"]].to_numpy()
lr_pick = LogisticRegression(max_iter=2000).fit(pick[idx_tr], y_all[idx_tr])
print(f"balanced accuracy from the P-pick index alone: {balanced_accuracy_score(y_all[idx_te], lr_pick.predict(pick[idx_te])):.3f}")
print(meta.groupby("source_type")["trace_P_arrival_sample"].agg(["min", "median", "max"]).astype(int))

Every surface event, sonic boom and thunder record has its "P pick" at sample 7000 exactly: for these sources the archive places the start of the signal at 70 s, while earthquake and explosion picks vary. A network that could see where the window sits in the original record would learn the label from bookkeeping. Our windows are cut around the pick, so this number never reaches the network. *Question:* name one other column of `meta` that would leak the label if you gave it to a model.

**(c) Choosing on the test set.** Train six versions (two learning rates, three widths). A careless report picks the one with the best **test** score. A careful one picks on the **validation** score, then reports the test score of that one.

In [ ]:
configs = [(lr, w) for lr in (1e-3, 3e-3) for w in (8, 16, 32)]
val_s, test_s = [], []
for lr, w in configs:
    m = build("cnn", seed=0, width=w)
    train(m, idx_tr, idx_va, lr=lr, epochs=10)
    val_s.append(score(m, Xs[idx_va], y_all[idx_va]))
    test_s.append(score(m, Xs[idx_te], y_all[idx_te]))
    print(f"lr {lr:g}, width {w:2d}: validation {val_s[-1]:.3f}  test {test_s[-1]:.3f}")

### Your turn (2): two reports

Write `reported_if_chosen_on_test` (the best test score of the six) and `reported_if_chosen_on_val` (the test score of the version with the best validation score). One line each, with `np.max`, `np.argmax` and the lists `test_s` and `val_s`.

In [ ]:
reported_if_chosen_on_test = None
reported_if_chosen_on_val = None

In [ ]:
check("reported_if_chosen_on_test", globals(), float(np.max(test_s)), "np.max(test_s)")
check("reported_if_chosen_on_val", globals(), float(test_s[int(np.argmax(val_s))]), "test_s[np.argmax(val_s)]")
print(f"\nchosen on test: {reported_if_chosen_on_test:.3f}   chosen on validation: {reported_if_chosen_on_val:.3f}   "
      f"difference: {reported_if_chosen_on_test - reported_if_chosen_on_val:+.3f}")

**Write your rule** (section 9) from the numbers of (a), (b) and (c). Compare the differences with the spread over seeds in (a): which differences are larger than the spread?

## 4. Station 2: starting weights

Notebook 4.0 started from random weights drawn with a seed and said "never zeros". Test it, and test weights that start too large.

*Prediction:* with every weight and bias at zero, what is the validation score after 12 epochs? With the default random weights multiplied by 10?

In [ ]:
my_prediction_2 = ""

### Your turn (3): two ways to start

Write `init_zeros(model)`, which sets every weight and bias of the model to zero, and `init_scaled(model, factor)`, which multiplies every weight and bias by `factor`. Both change the model in place inside `with torch.no_grad():`, looping over `model.parameters()`. Hint: a tensor `p` has the methods `p.zero_()` and `p.mul_(factor)`.

In [ ]:
init_zeros = None
init_scaled = None

In [ ]:
def _init_zeros(model):
    with torch.no_grad():
        for p in model.parameters():
            p.zero_()


def _init_scaled(model, factor):
    with torch.no_grad():
        for p in model.parameters():
            p.mul_(factor)


def _test_zeros(f):
    m = build("cnn", seed=0); f(m)
    return all(float(p.abs().max()) == 0 for p in m.parameters())


def _test_scaled(f):
    a, b = build("cnn", seed=0), build("cnn", seed=0); f(b, 10.0)
    return all(torch.allclose(10 * p, q) for p, q in zip(a.parameters(), b.parameters()))


use_or_reference("init_zeros", globals(), _init_zeros, _test_zeros, "for p in model.parameters(): p.zero_()")
use_or_reference("init_scaled", globals(), _init_scaled, _test_scaled, "for p in model.parameters(): p.mul_(factor)")

res2, hist2 = {}, {}
for label, prep in [("seeded random (default)", lambda m: None), ("all zeros", init_zeros),
                    ("random x 10", lambda m: init_scaled(m, 10.0)), ("random x 0.1", lambda m: init_scaled(m, 0.1))]:
    s = []
    for seed in range(3):
        m = build("cnn", seed=seed); prep(m)
        h = train(m, idx_tr, idx_va, seed=seed)
        s.append(h["val_score"][-1])
        if seed == 0:
            hist2[label] = h
    res2[label] = s
    print(f"{label:24s} validation balanced accuracy {np.mean(s):.3f} ± {np.std(s):.3f}  (3 seeds)")
plot_curves(hist2, "Starting weights, seed 0")

*Questions.* With zero starting weights, how many different output scores does the network give over the validation records? (Try `np.unique(predict(m, Xs[idx_va]))` on a zero-started model.) Why does a hidden unit that starts equal to its neighbor stay equal to it? Is a balanced accuracy of 0.2 better than chance for five classes?

**Write your rule** (section 9).

## 5. Station 3: learning rate and batch size

The learning rate sets the size of each update; the batch size sets how many records are averaged in the loss before each update. They act together: a smaller batch means more updates per epoch, each noisier.

*Prediction:* of learning rates 0.0001, 0.001 and 0.01 with Adam, which trains best in 8 epochs? Does the answer change with a batch of 16 instead of 256?

In [ ]:
my_prediction_3 = ""

### Your turn (4): the grid

Write the double loop: for each learning rate in `lrs` and each batch size in `batches`, build a CNN with `build("cnn", seed=0)`, train it with `train(m, idx_tr, idx_va, lr=lr, batch=b, epochs=8)`, and store the returned history in `grid[(lr, b)]`. Four lines.

In [ ]:
lrs = [1e-4, 1e-3, 1e-2]
batches = [16, 256]
grid = {}
# your loop here

In [ ]:
if len(grid) != len(lrs) * len(batches):
    print("grid: incomplete. Hint: for lr in lrs: for b in batches: m = build(...); grid[(lr, b)] = train(...)."
          "\n  Running the reference loop.")
    grid = {}
    for lr in lrs:
        for b in batches:
            m = build("cnn", seed=0)
            grid[(lr, b)] = train(m, idx_tr, idx_va, lr=lr, batch=b, epochs=8)

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, b in zip(axes, batches):
    plot_curves({f"lr {lr:g}": grid[(lr, b)] for lr in lrs}, f"batch {b}: {grid[(lrs[0], b)]['updates']} updates in 8 epochs", ax)
fig.tight_layout()
print("validation balanced accuracy after 8 epochs (best epoch kept):")
print(pd.DataFrame({f"batch {b}": [max(grid[(lr, b)]["val_score"]) for lr in lrs] for b in batches},
                   index=[f"lr {lr:g}" for lr in lrs]).round(3))

**A fair comparison.** Batch 16 made far more updates than batch 256 in the same 8 epochs. Train both for the **same number of updates** (`max_updates=200`) and look again: which batch size wins now? The noise of the loss from batch to batch is in `h["batch_loss"]`; plot it for both.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
for b in batches:
    m = build("cnn", seed=0)
    h = train(m, idx_tr, idx_va, lr=1e-3, batch=b, epochs=200, max_updates=200)
    ax.plot(h["batch_loss"], lw=1, label=f"batch {b}: validation {max(h['val_score']):.3f}")
ax.set_xlabel("update"); ax.set_ylabel("loss of the batch"); ax.legend(); ax.set_title("200 updates each", loc="left")

**Write your rule** (section 9).

## 6. Station 4: model size and amount of data

Two ways to improve a network: give it more weights, or give it more records. A **learning curve** answers which: the score against the number of training records, for each size of network.

*Prediction:* with 25% of the training records, does the biggest network do better or worse than the smallest?

In [ ]:
my_prediction_4 = ""

### Your turn (5): a smaller training set

Write `subsample(idx, fraction, seed)`, which returns a random subset of the record indices `idx` of size `int(fraction * len(idx))`, without repeats. Use `np.random.default_rng(seed).choice(...)` with `replace=False`.

In [ ]:
subsample = None

In [ ]:
def _subsample(idx, fraction, seed):
    return np.random.default_rng(seed).choice(idx, int(fraction * len(idx)), replace=False)


use_or_reference("subsample", globals(), _subsample,
                 lambda f: len(set(f(idx_tr, 0.25, 0))) == int(0.25 * len(idx_tr)) and set(f(idx_tr, 0.25, 0)) <= set(idx_tr),
                 "np.random.default_rng(seed).choice(idx, int(fraction * len(idx)), replace=False)")

fractions, widths = [0.25, 0.5, 1.0], [4, 16, 48]
res4 = pd.DataFrame(index=[f"{f:.0%} of train" for f in fractions])
for w in widths:
    col = []
    for fr in fractions:
        s = []
        for seed in range(2):
            m = build("cnn", seed=seed, width=w)
            train(m, subsample(idx_tr, fr, seed), idx_va, seed=seed, epochs=12)
            s.append(score(m, Xs[idx_va], y_all[idx_va]))
        col.append(np.mean(s))
    res4[f"width {w} ({n_weights(m):,} weights)"] = col
print("validation balanced accuracy, mean of 2 seeds")
print(res4.round(3))
ax = res4.set_index(pd.Index([int(f * len(idx_tr)) for f in fractions])).plot(marker="o", figsize=(7, 4))
ax.set_xlabel("training records"); ax.set_ylabel("validation balanced accuracy"); ax.set_title("Learning curves", loc="left")

**Fully connected against convolutional.** The fully connected network sees 90 numbers per record (log RMS amplitude per second and component), not the waveform. Train it and compare its score and its number of weights with the CNN of section 2.

In [ ]:
fcn = build("fcn", seed=0, width=64)
hf = train(fcn, idx_tr, idx_va, epochs=30)
print(f"FCN: {n_weights(fcn):,} weights, test balanced accuracy {score(fcn, Xs[idx_te], y_all[idx_te]):.3f}")
print(f"CNN: {n_weights(cnn):,} weights, test balanced accuracy {score(cnn, Xs[idx_te], y_all[idx_te]):.3f}")
plot_curves({"FCN": hf}, "Fully connected network on the 90 RMS values")

*Questions.* Would you collect more records or build a bigger network? What does the FCN know that the CNN has to learn, and what does it throw away?

**Write your rule** (section 9).

## 7. Station 5: data augmentation

Every training window starts 5 s before the P pick. A network running on continuous data will see arrivals anywhere in its window, and on a noisier day. Augmentation changes each training batch in ways that keep the label: here, a random shift in time, a random sign (ground motion and its opposite come from the same kind of source), and real background noise from the records' own pre-event windows.

First, the test that matters: the same test records with the arrival moved by up to 10 s.

In [ ]:
def shift_records(X, max_shift_s, seed):
    g = np.random.default_rng(seed)
    out = np.empty_like(X)
    for i in range(len(X)):
        out[i] = np.roll(X[i], int(g.integers(-max_shift_s * FS, max_shift_s * FS + 1)), axis=-1)
    return out

X_te_shift = shift_records(Xs[idx_te], 10, seed=1)
print(f"reference CNN, test balanced accuracy: as cut {score(cnn, Xs[idx_te], y_all[idx_te]):.3f}, "
      f"arrival moved up to ±10 s {score(cnn, X_te_shift, y_all[idx_te]):.3f}")

*Prediction:* will training with random shifts close the gap? Will it cost anything on the windows as cut?

In [ ]:
my_prediction_5 = ""

### Your turn (6): an augmentation

Write `augment(xb)`. It receives a batch, a tensor of shape (records, 3, 1500), and returns a batch of the same shape where each record is (1) moved in time by a random number of samples between −500 and 500 (±10 s) with `torch.roll(x, shifts=k, dims=-1)`, and (2) multiplied by +1 or −1 at random. Use `torch.randint` for both.

In [ ]:
augment = None

In [ ]:
def _augment(xb):
    out = torch.empty_like(xb)
    k = torch.randint(-500, 501, (len(xb),))
    sign = torch.randint(0, 2, (len(xb),)) * 2 - 1
    for i in range(len(xb)):
        out[i] = sign[i] * torch.roll(xb[i], int(k[i]), dims=-1)
    return out


def _test_aug(f):
    xb = torch.from_numpy(Xs[:8].copy())
    o = f(xb.clone())
    same_energy = torch.allclose((o ** 2).sum(dim=(1, 2)), (xb ** 2).sum(dim=(1, 2)), rtol=1e-4)
    return o.shape == xb.shape and same_energy and not torch.allclose(o, xb)


use_or_reference("augment", globals(), _augment, _test_aug,
                 "loop over records: sign * torch.roll(x, int(k), dims=-1), with k and sign from torch.randint")

noise_bank = torch.from_numpy(Ns[idx_tr])        # background noise from TRAINING records only


def augment_noise(xb, level=1.0):
    xb = augment(xb)
    j = torch.randint(0, len(noise_bank), (len(xb),))
    return xb + level * torch.rand(len(xb), 1, 1) * noise_bank[j]


res5 = {}
for label, aug in [("no augmentation", None), ("shift and sign", augment), ("shift, sign and real noise", augment_noise)]:
    a, b = [], []
    for seed in range(2):
        m = build("cnn", seed=seed)
        train(m, idx_tr, idx_va, augment=aug, seed=seed, epochs=15)
        a.append(score(m, Xs[idx_te], y_all[idx_te])); b.append(score(m, X_te_shift, y_all[idx_te]))
    res5[label] = (np.mean(a), np.mean(b))
print(pd.DataFrame(res5, index=["test as cut", "test, arrival moved ±10 s"]).T.round(3), "\n(mean of 2 seeds)")

*Questions.* The noise bank uses the pre-event windows of the **training** records only. What would go wrong if it also used the test records' noise? Which augmentation would be wrong for this task (it changes the label, or makes records no instrument would record)?

**Write your rule** (section 9).

## 8. Station 6: the loss

Cross-entropy treats every record alike, so the 500 earthquakes outweigh the 94 thunder records. Four losses, one question: how many sonic booms and thunder records does the network find (their recall)?

- **cross-entropy**, as in every station so far;
- **weighted cross-entropy**: the loss of each record multiplied by a weight for its class, the inverse of the class's share of the training set;
- **focal loss** (Lin et al., 2017): the loss of each record multiplied by $(1 - p)^\gamma$, where $p$ is the probability the network gives to the right class, so records already classified with confidence count less;
- **label smoothing**: the target is 0.9 for the right class and 0.025 for each other class, for labels that analysts sometimes get wrong.

*Prediction:* which loss gives the highest recall on thunder? Which gives the highest recall on earthquakes?

In [ ]:
my_prediction_6 = ""

### Your turn (7): the focal loss

Write `focal_loss(logits, target, gamma=2.0)`. Steps: `logp = F.log_softmax(logits, dim=1)`; the log-probability of the right class `logp_true = logp.gather(1, target[:, None])[:, 0]`; `p_true = logp_true.exp()`; the loss of each record is `-(1 - p_true) ** gamma * logp_true`; return its mean over the batch.

In [ ]:
focal_loss = None

In [ ]:
def _focal(logits, target, gamma=2.0):
    logp = F.log_softmax(logits, dim=1)
    logp_true = logp.gather(1, target[:, None])[:, 0]
    return (-(1 - logp_true.exp()) ** gamma * logp_true).mean()


def _test_focal(f):
    lg, tg = torch.randn(32, 5), torch.randint(0, 5, (32,))
    return torch.allclose(f(lg, tg), _focal(lg, tg), atol=1e-5) and torch.allclose(f(lg, tg, gamma=0.0), F.cross_entropy(lg, tg), atol=1e-5)


use_or_reference("focal_loss", globals(), _focal, _test_focal, "follow the four steps; gamma = 0 must give cross-entropy")

counts = np.bincount(y_all[idx_tr], minlength=5)
w_class = torch.tensor(len(idx_tr) / (5 * counts), dtype=torch.float32)
print("class weights:", dict(zip(CLASSES, w_class.numpy().round(2))))
losses = {"cross-entropy": nn.CrossEntropyLoss(), "weighted cross-entropy": nn.CrossEntropyLoss(weight=w_class),
          "focal, gamma 2": focal_loss, "label smoothing 0.1": nn.CrossEntropyLoss(label_smoothing=0.1)}
rows = {}
for label, lf in losses.items():
    rec = []
    for seed in range(2):
        m = build("cnn", seed=seed)
        train(m, idx_tr, idx_va, loss_fn=lf, seed=seed, epochs=15)
        rec.append(recall_score(y_all[idx_te], predict(m, Xs[idx_te]), average=None, labels=list(range(5)), zero_division=0))
    rows[label] = np.mean(rec, axis=0)
res6 = pd.DataFrame(rows, index=[f"recall {c}" for c in CLASSES]).T
res6["balanced accuracy"] = res6.mean(axis=1)
print(res6.round(2), "\n(test set, mean of 2 seeds)")

*Your own loss.* Write a loss that encodes a cost you care about, and train with it (copy the cell above, keep one row). Two ideas: a missed explosion costs a monitoring agency more than an explosion called an earthquake, so penalize that one error more (a cost matrix applied to the predicted probabilities); or combine cross-entropy with the focal term. State the cost in physical or operational terms before you write the code.

**Write your rule** (section 9).

## 9. Your rules

Write one rule per station you ran, in this format. A rule says what to do, how to check it, and the number from your run that justifies it. These go on the class board; at the end of the session we compare them with the course rubric.

| Station | Rule (what to do) | Check (how someone else verifies it) | Evidence (your number, with seeds) |
|---|---|---|---|
| | | | |
| | | | |

Example of the format, from notebook 4.1: *"Scale the inputs with statistics of the training records only. Check: the line that fits the scaler. Evidence: the scaler is fit on `X_train` before the validation and test sets are transformed."*

## 10. Reflection

Answer in two or three sentences each.

1. **What you learned.** Which result of today surprised you most, and what did you expect instead?
2. **What you learned from the agents.** Your team's coding agent built a detector from one prompt while you ran the stations. Which of today's decisions did it make well, which did it skip, and how would you have known without running the stations yourself?
3. **How to improve it.** Which station would you change, or which decision is missing from the six, to make this clinic more useful for your project?